In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
import re
import dgl
from torch.utils.data import DataLoader
import math 
import torch.nn.functional as F
import dgl.function as fn
from dgl.nn.functional import edge_softmax

F:\anaconda\envs\wppenv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Thunderbird_log = pd.read_csv(r"G:\wpp\dataset and paper\Transfer-HGT\data-process\Thunderbird\Thunderbird_log_structured.csv",memory_map=True)
Thunderbird_log.head()

,LineId,Label,Timestamp,Date,User,Month,Day,Time,Location,Component,PID,Content,EventId,EventTemplate,ParameterList,Key
0,600001,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
1,600002,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
2,600003,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
3,600004,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
4,600005,"""VAPI",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367


In [3]:
Thunderbird_log['Content'][0],Thunderbird_log['EventTemplate'][0]

('THH(1): mnt_projects/sysapps/src/ib/topspin/topspin-src-3.2.0-16/third_party/thca4_linux/kernel/mlxhh/thh/obj_host_amd64_custom1_rhel4/mod_thh/hob_comm.c[1597]: XHH_hob_process_local_mad: Device in FATAL state',
 'THH(1): <*> <*> Device in FATAL state')

In [4]:
BGL_Log_Structured = pd.read_csv(r"G:\wpp\dataset and paper\Transfer-HGT\data\BGL\BGL_Log_Structured.csv",memory_map=True)
BGL_Log_Structured.head()

,LineId,Label,Timestamp,Date,Node,Time,NodeRepeat,Type,Component,Level,Content,EventId,EventTemplate,ParameterList
0,1,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.363779,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
1,2,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.527847,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
2,3,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
3,4,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.823719,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
4,5,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.982731,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]


In [7]:
BGL_Log_Structured['Content'][10000],BGL_Log_Structured['EventTemplate'][10000]

('generating core.1038', 'generating <*>')

In [3]:
Thunderbird_log

,LineId,Label,Timestamp,Date,User,Month,Day,Time,Location,Component,PID,Content,EventId,EventTemplate,ParameterList,Key
0,600001,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
1,600002,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
2,600003,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
3,600004,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
4,600005,"""VAPI",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,kernel,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399526,999527,"""-",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
399527,999528,"""VAPI",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,kernel,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
399528,999529,"""VAPI",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,kernel,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
399529,999530,"""-",1131963039,2005.11.14,dn30,Nov,14,02:10:39,dn30/dn30,kernel,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395


In [4]:
'''
存在不同组件，相同的日志事件   的情况
'''
com = Thunderbird_log['Component'].unique()
com_id=np.array([x for x in range(1,55) ])
com_dict = dict(zip(com,com_id))
# com_dict
# for index,row in Thunderbird_log.iterrows():
#     if row['Component']=='portmap':
#         print(row)       

In [5]:
for i in com_dict.keys():
    com_dict[i] = "C"+str(com_dict[i])
com_dict

{'kernel': 'C1',
 'ntpd': 'C2',
 'ib_sm.x': 'C3',
 '/apps/x86_64/system/ganglia-3.0.1/sbin/gmetad': 'C4',
 'Server Administrator': 'C5',
 'snmpd': 'C6',
 'syslog-ng': 'C7',
 'crond(pam_unix)': 'C8',
 'crond': 'C9',
 'dhcpd': 'C10',
 'xinetd': 'C11',
 'sendmail': 'C12',
 'pam_limits': 'C13',
 'irqbalance': 'C14',
 'syslog': 'C15',
 'syslogd 1.4.1': 'C16',
 'nfslock': 'C17',
 'portmap': 'C18',
 'rc': 'C19',
 'rpc.statd': 'C20',
 'rpcidmapd': 'C21',
 'ts_fixup': 'C22',
 'netfs': 'C23',
 'acpid': 'C24',
 'date': 'C25',
 'fsck': 'C26',
 'init': 'C27',
 'lvm.static': 'C28',
 'mount.panfs': 'C29',
 'mount': 'C30',
 'network': 'C31',
 'rc.sysinit': 'C32',
 'smartd': 'C33',
 'start_udev': 'C34',
 'sysctl': 'C35',
 'udevsend': 'C36',
 'instsvcdrv': 'C37',
 'dataeng': 'C38',
 'sshd': 'C39',
 'gmond': 'C40',
 'ntpdate': 'C41',
 'pbs_mom': 'C42',
 'messagebus': 'C43',
 'omsad': 'C44',
 'fstab-sync': 'C45',
 'haldaemon': 'C46',
 'sudo': 'C47',
 'logger': 'C48',
 'userhelper': 'C49',
 'sshd(pam_unix)

In [6]:
Thunderbird_log['Component'] = Thunderbird_log['Component'].apply( lambda x: com_dict[x])
Thunderbird_log

,LineId,Label,Timestamp,Date,User,Month,Day,Time,Location,Component,PID,Content,EventId,EventTemplate,ParameterList,Key
0,600001,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
1,600002,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
2,600003,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
3,600004,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
4,600005,"""VAPI",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399526,999527,"""-",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
399527,999528,"""VAPI",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,C1,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
399528,999529,"""VAPI",1131963039,2005.11.14,an933,Nov,14,02:10:39,an933/an933,C1,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
399529,999530,"""-",1131963039,2005.11.14,dn30,Nov,14,02:10:39,dn30/dn30,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395


In [7]:
Event = Thunderbird_log.drop_duplicates(subset='EventId',inplace=False)
Event

,LineId,Label,Timestamp,Date,User,Month,Day,Time,Location,Component,PID,Content,EventId,EventTemplate,ParameterList,Key
0,600001,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
2,600003,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
4,600005,"""VAPI",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
10,600011,"""-",1131904443,2005.11.13,dn25,Nov,13,09:54:03,dn25/dn25,C2,2260.0,"synchronized to 10.100.28.250, stratum 3""",ef226888,synchronized to <*> stratum <*>,"['10.100.28.250,', '3""']",E44
24,600025,-,1131904444,2005.11.13,tbird-sm1,Nov,13,09:54:04,src@tbird-sm1,C3,24904.0,[INFO]: Configuration caused by some ports in ...,5d37b30c,[INFO]: Configuration caused by some ports in ...,[],E375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315666,915667,-,1131951605,2005.11.13,bn575,Nov,13,23:00:05,bn575/bn575,C26,NaN,[/sbin/fsck.ext3 (1) -- /var],24bc2b8c,[/sbin/fsck.ext3 (1) -- <*>,['/var]'],E407
315667,915668,-,1131951605,2005.11.13,bn575,Nov,13,23:00:05,bn575/bn575,C26,NaN,fsck.ext3 -a /dev/sda5,f5db9ad3,fsck.ext3 -a <*>,['/dev/sda5'],E408
331143,931144,-,1131953731,2005.11.13,bn575,Nov,13,23:35:31,bn575/bn575,C25,NaN,Mon Nov 14 00:35:16 MST 2005,b5547c6f,Mon Nov 14 <*> MST 2005,['00:35:16'],E14
336911,936912,-,1131954540,2005.11.13,#1#,Nov,13,23:49:00,#1#/#1#,C1,NaN,audit(1131950940.033:0): avc: denied { search ...,27c493db,audit(<*><*>): avc: denied { search } for pid<...,"['1131950940.033:0', '=19258', '=30 scontext=u...",E402


In [8]:
com_dict
for i in com_dict.values(): 
    print(i)

C1
C2
C3
C4
C5
C6
C7
C8
C9
C10
C11
C12
C13
C14
C15
C16
C17
C18
C19
C20
C21
C22
C23
C24
C25
C26
C27
C28
C29
C30
C31
C32
C33
C34
C35
C36
C37
C38
C39
C40
C41
C42
C43
C44
C45
C46
C47
C48
C49
C50
C51
C52
C53
C54


In [9]:
com_Event_dict = OrderedDict()
for i in com_dict.values(): 
    if i not in com_Event_dict:
        com_Event_dict[i] = []
for index,row in  Thunderbird_log.iterrows():
    if row['EventId'] not in com_Event_dict[row['Component']]:
        com_Event_dict[row['Component']].append(row['EventId'])
com_Event_dict

OrderedDict([('C1',
              ['f29fb486',
               'afa7654c',
               '4f5b9781',
               '45d319cb',
               'a1cbc82b',
               'a20abb3d',
               '6dc36b09',
               'b6bf46de',
               '0f5f997d',
               'a4ff4c6f',
               'f69dda74',
               'ad416b7e',
               'af614352',
               '87cf408f',
               '71357687',
               '115635d6',
               '295c8236',
               '6204d307',
               '8c537990',
               'f3baae88',
               '7e2eac94',
               'de6cd9cf',
               '6d58ac7e',
               'c36d1d3d',
               '1d743fc4',
               '7d2f04f5',
               'cc2cb575',
               'c1a59557',
               '5f45a05e',
               'cd56675f',
               '2fec979e',
               'ab8e12aa',
               '1cf019dc',
               'e727dd7e',
               'f0988321',
               '1ce35610',
        

In [10]:
l = []
for index,row in Thunderbird_log.iterrows():
    l.append(com_Event_dict[row['Component']].index(row['EventId']))
Thunderbird_log["id"] = l

In [11]:
Thunderbird_log.drop(labels=['Date','Time','PID','User','Month','Day','Location','PID','ParameterList','Timestamp','Content'],axis=1, inplace=True)
Thunderbird_log

,LineId,Label,Component,EventId,EventTemplate,Key,id
0,600001,-,C1,f29fb486,THH(1): <*> <*> Device in FATAL state,E52,0
1,600002,-,C1,f29fb486,THH(1): <*> <*> Device in FATAL state,E52,0
2,600003,"""-",C1,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
3,600004,"""-",C1,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
4,600005,"""VAPI",C1,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
...,...,...,...,...,...,...,...
399526,999527,"""-",C1,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
399527,999528,"""VAPI",C1,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
399528,999529,"""VAPI",C1,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
399529,999530,"""-",C1,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1


In [12]:
EventId_dict = dict(zip(Thunderbird_log['EventId'],Thunderbird_log['id']))
EventId_dict

{'f29fb486': 0,
 'afa7654c': 1,
 '4f5b9781': 2,
 'ef226888': 0,
 '5d37b30c': 0,
 '8e6e114b': 1,
 '7861dafa': 2,
 '51a56721': 3,
 '0170c871': 4,
 '6391d957': 0,
 '7af5f6c5': 0,
 '5f6a681c': 0,
 'b523b147': 5,
 'a0939195': 6,
 '8d134961': 7,
 '5b2bcc81': 1,
 '50345058': 0,
 '09789b2b': 0,
 '5947eee3': 0,
 '27624832': 1,
 'f4455acc': 0,
 'd0052946': 1,
 '3b327ac5': 2,
 'a82970ae': 3,
 'e8b0f746': 4,
 'fe7d0745': 5,
 'f8c00723': 0,
 '182a3d2f': 0,
 '910e288d': 0,
 'b10a98bd': 1,
 '2315719c': 0,
 '45d319cb': 3,
 'a1cbc82b': 4,
 'a20abb3d': 5,
 '6dc36b09': 6,
 'b6bf46de': 7,
 '0f5f997d': 8,
 'a4ff4c6f': 9,
 'f69dda74': 10,
 'ad416b7e': 11,
 'af614352': 12,
 '87cf408f': 13,
 '71357687': 14,
 '115635d6': 15,
 '295c8236': 16,
 '6204d307': 17,
 '8c537990': 18,
 'f3baae88': 19,
 '7e2eac94': 20,
 'de6cd9cf': 21,
 '6d58ac7e': 22,
 'c36d1d3d': 23,
 '1d743fc4': 24,
 '7d2f04f5': 25,
 'cc2cb575': 26,
 'c1a59557': 27,
 '5f45a05e': 28,
 'cd56675f': 29,
 '2fec979e': 30,
 'ab8e12aa': 31,
 '1cf019dc': 32,
 

In [13]:
Thunderbird_log['EventId'] = Thunderbird_log['EventId'].apply( lambda x: EventId_dict[x])
Thunderbird_log

,LineId,Label,Component,EventId,EventTemplate,Key,id
0,600001,-,C1,0,THH(1): <*> <*> Device in FATAL state,E52,0
1,600002,-,C1,0,THH(1): <*> <*> Device in FATAL state,E52,0
2,600003,"""-",C1,1,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
3,600004,"""-",C1,1,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
4,600005,"""VAPI",C1,2,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
...,...,...,...,...,...,...,...
399526,999527,"""-",C1,1,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1
399527,999528,"""VAPI",C1,2,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
399528,999529,"""VAPI",C1,2,"<*> <*> failed, return code = -254 (Fatal erro...",E367,2
399529,999530,"""-",C1,1,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,E395,1


In [14]:
Event

,LineId,Label,Timestamp,Date,User,Month,Day,Time,Location,Component,PID,Content,EventId,EventTemplate,ParameterList,Key
0,600001,-,1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,THH(1): mnt_projects/sysapps/src/ib/topspin/to...,f29fb486,THH(1): <*> <*> Device in FATAL state,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E52
2,600003,"""-",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][ib_mad_dispatch][/mnt_projects/sys...,afa7654c,[KERNEL_IB][ib_mad_dispatch][/<*>-<*>-16/ib/ts...,['mnt_projects/sysapps/src/ib/topspin/topspin-...,E395
4,600005,"""VAPI",1131904443,2005.11.13,an416,Nov,13,09:54:03,an416/an416,C1,NaN,[KERNEL_IB][tsIbTavorMadProcess][/mnt_projects...,4f5b9781,"<*> <*> failed, return code = -254 (Fatal erro...",['[KERNEL_IB][tsIbTavorMadProcess][/mnt_projec...,E367
10,600011,"""-",1131904443,2005.11.13,dn25,Nov,13,09:54:03,dn25/dn25,C2,2260.0,"synchronized to 10.100.28.250, stratum 3""",ef226888,synchronized to <*> stratum <*>,"['10.100.28.250,', '3""']",E44
24,600025,-,1131904444,2005.11.13,tbird-sm1,Nov,13,09:54:04,src@tbird-sm1,C3,24904.0,[INFO]: Configuration caused by some ports in ...,5d37b30c,[INFO]: Configuration caused by some ports in ...,[],E375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315666,915667,-,1131951605,2005.11.13,bn575,Nov,13,23:00:05,bn575/bn575,C26,NaN,[/sbin/fsck.ext3 (1) -- /var],24bc2b8c,[/sbin/fsck.ext3 (1) -- <*>,['/var]'],E407
315667,915668,-,1131951605,2005.11.13,bn575,Nov,13,23:00:05,bn575/bn575,C26,NaN,fsck.ext3 -a /dev/sda5,f5db9ad3,fsck.ext3 -a <*>,['/dev/sda5'],E408
331143,931144,-,1131953731,2005.11.13,bn575,Nov,13,23:35:31,bn575/bn575,C25,NaN,Mon Nov 14 00:35:16 MST 2005,b5547c6f,Mon Nov 14 <*> MST 2005,['00:35:16'],E14
336911,936912,-,1131954540,2005.11.13,#1#,Nov,13,23:49:00,#1#/#1#,C1,NaN,audit(1131950940.033:0): avc: denied { search ...,27c493db,audit(<*><*>): avc: denied { search } for pid<...,"['1131950940.033:0', '=19258', '=30 scontext=u...",E402


In [15]:
Thunderbird_log["EventTemplate"] = Thunderbird_log["EventTemplate"].map(lambda x: re.sub("[<*>/:]","",x))
Thunderbird_log["EventTemplate"] = Thunderbird_log["EventTemplate"].map(lambda x: "[CLS] " + x + " [SEP]")
Thunderbird_log

,LineId,Label,Component,EventId,EventTemplate,Key,id
0,600001,-,C1,0,[CLS] THH(1) Device in FATAL state [SEP],E52,0
1,600002,-,C1,0,[CLS] THH(1) Device in FATAL state [SEP],E52,0
2,600003,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
3,600004,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
4,600005,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
...,...,...,...,...,...,...,...
399526,999527,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
399527,999528,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
399528,999529,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
399529,999530,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1


In [16]:
from transformers import logging
from transformers import BertModel, BertTokenizer
logging.set_verbosity_warning()
model = BertModel.from_pretrained('bert-base-uncased')

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0): BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          

In [19]:
def sent_to_vec(sent, tokenizer, model, pooling, max_length):
    with torch.no_grad():
        inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True,  max_length=max_length)
        inputs['input_ids'] = inputs['input_ids'].to(device)
        inputs['token_type_ids'] = inputs['token_type_ids'].to(device)
        inputs['attention_mask'] = inputs['attention_mask'].to(device)

        hidden_states = model(**inputs, return_dict=True, output_hidden_states=True).hidden_states
        output_hidden_state = (hidden_states[-1] + hidden_states[1]).mean(dim=1)
        vec = output_hidden_state.cpu().numpy()[0]
    return vec

In [20]:
def sents_to_vecs(sents, tokenizer, model, pooling, max_length, verbose=True):
    vecs = []
    if verbose:
        sents = tqdm(sents)
    for sent in sents:
        vec = sent_to_vec(sent, tokenizer, model, pooling, max_length)
        vecs.append(vec)
    assert len(sents) == len(vecs)
    vecs = np.array(vecs)
    return vecs

In [21]:
def compute_kernel_bias(vecs):
    """计算kernel和bias
    最后的变换：y = (x + bias).dot(kernel)
    """
    mu = vecs.mean(axis=0, keepdims=True)#(1, 768) 求各列的均值
    cov = np.cov(vecs.T) #(768,768)计算每个特征维度之间的协方差
    u, s, vh = np.linalg.svd(cov) #进行奇异值分解
    W_inv = np.dot(u, np.diag(s**0.5)) 
    W = np.linalg.inv(W_inv.T)
    return W[:, :256], -mu

In [22]:
def transform_and_normalize(vecs, kernel=None, bias=None):
    """应用变换，然后标准化
    """
    if not (kernel is None or bias is None):
        vecs = (vecs + bias).dot(kernel)
    return vecs / (vecs**2).sum(axis=1, keepdims=True)**0.5

In [23]:
EventTemplate = Event['EventTemplate']

In [ ]:
from tqdm import tqdm
data_vec = sents_to_vecs(EventTemplate, tokenizer, model, 'first_last_avg', 64)

 61%|█████████████████████████████████████████████████▏                              | 226/368 [00:09<00:05, 24.69it/s]

In [ ]:
kernel, bias = compute_kernel_bias(data_vec)

In [45]:
load_dict = np.load('com_encode_dict256tb.npy',allow_pickle=True).item()

In [142]:
Thunderbird_log

,LineId,Label,Component,EventId,EventTemplate,Key,id
0,600001,-,C1,0,[CLS] THH(1) Device in FATAL state [SEP],E52,0
1,600002,-,C1,0,[CLS] THH(1) Device in FATAL state [SEP],E52,0
2,600003,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
3,600004,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
4,600005,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
...,...,...,...,...,...,...,...
399526,999527,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1
399527,999528,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
399528,999529,"""VAPI",C1,2,"[CLS] failed, return code = -254 (Fatal erro...",E367,2
399529,999530,"""-",C1,1,[CLS] [KERNEL_IB][ib_mad_dispatch][--16ibts_ap...,E395,1


In [143]:
log_data = []
sequence_len=20
for i in range(0,len(Thunderbird_log)-sequence_len,10):
    log_data.append(Thunderbird_log['EventId'][i:i+sequence_len])
reshaped_logs = np.array(log_data).astype('float64')
reshaped_log=[]
for i in range(len(reshaped_logs)):
    reshaped_log.append(list(map(int,reshaped_logs[i])))
reshaped_log

[[0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1],
 [0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 2, 0, 0, 1, 1, 1, 2, 2],
 [2, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 0, 0, 1, 1, 2],
 [2, 3, 3, 3, 4, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 1, 2, 2, 0, 0],
 [2, 0, 0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 0, 0, 0, 1, 1, 2, 2],
 [5, 6, 7, 0, 0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 0, 0, 0, 1],
 [0, 0, 1, 2, 0, 0, 0, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 1, 2, 2],
 [1, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 0, 0, 0, 0, 0, 1, 1, 1, 2],
 [2, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 0, 1, 1, 2],
 [2, 2, 3, 3, 3, 4, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 0, 0, 0],
 [2, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 5, 6, 7, 0, 0, 0, 1],
 [0, 0, 0, 5, 6, 7, 0, 0, 0, 1, 2, 0, 0, 1, 2, 0, 0, 0, 1, 1],
 [2, 0, 0, 1, 2, 0, 0, 0, 1, 1, 1, 2, 2, 2, 0, 0, 1, 1,

In [157]:
com_data = []
sequence_len=20
for j in range(0,len(Thunderbird_log)-sequence_len,10):
    com_data.append(Thunderbird_log['Component'][j:j+sequence_len])
reshaped_com = np.array(com_data).astype('str')
reshaped_com[0:10]

array([['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C2',
        'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1'],
       ['C2', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1',
        'C1', 'C1', 'C2', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3'],
       ['C1', 'C1', 'C1', 'C2', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3',
        'C3', 'C3', 'C3', 'C3', 'C1', 'C1', 'C1', 'C1', 'C1'],
       ['C3', 'C3', 'C3', 'C3', 'C3', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1',
        'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C4', 'C4'],
       ['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C4', 'C4', 'C4',
        'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C2', 'C5'],
       ['C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C2', 'C5', 'C6',
        'C2', 'C2', 'C4', 'C4', 'C4', 'C4', 'C4', 'C2', 'C4'],
       ['C6', 'C2', 'C2', 'C4', 'C4', 'C4', 'C4', 'C4', 'C2', 'C4', 'C3',
        'C3', 'C3', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1'],
       ['C3', 'C3', 'C3', 'C1', 'C1', 'C1

In [106]:
# log_label = []
# for k in range(0,len(Thunderbird_log)-sequence_len,10):
#     log_label.append(Thunderbird_log['Label'][i:i+sequence_len])
# reshaped_label = np.array(log_label).astype('str')
# print(reshaped_label)
# label=[]
# for i in range(len(reshaped_label)):
#      for j in range(len(reshaped_label[i])):
#             if reshaped_label[i][j]=='"VAPI':
#                 label.append(1)
#                 break
#             else:
#                 label.append(0)
# a=0
# for i in range(len(label)):
#     if label[i]==0:
#         a = a+1

# print(a)

[['"VAPI' '-' '-' ... '-' '-' '-']
 ['"VAPI' '-' '-' ... '-' '-' '-']
 ['"VAPI' '-' '-' ... '-' '-' '-']
 ...
 ['"VAPI' '-' '-' ... '-' '-' '-']
 ['"VAPI' '-' '-' ... '-' '-' '-']
 ['"VAPI' '-' '-' ... '-' '-' '-']]


In [73]:
# reshaped_label=lambda x:1 if reshaped_label[x]=='"VAPI' else 0
# reshaped_label

<function __main__.<lambda>(x)>

In [152]:
# reshaped_log=reshaped_log.tolist()
# data_dict = pd.DataFrame()
s=pd.Series(reshaped_log)
s

0        [0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1, 1, 2, ...
1        [0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 2, 0, 0, ...
2        [2, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, ...
3        [2, 3, 3, 3, 4, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...
4        [2, 0, 0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, ...
                               ...                        
39947    [0, 1, 1, 2, 2, 1, 0, 0, 0, 1, 1, 2, 2, 1, 2, ...
39948    [1, 2, 2, 1, 2, 3, 0, 0, 0, 0, 1, 1, 1, 2, 2, ...
39949    [1, 1, 1, 2, 2, 2, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...
39950    [2, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 1, 1, 1, 2, ...
39951    [0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 1, 2, 0, 0, ...
Length: 39952, dtype: object

In [151]:
# reshaped_com=reshaped_com.tolist()
df_dict = pd.DataFrame(s, columns=['EventSequence'])
df_dict['Component']=reshaped_com
df_dict

,EventSequence,Component
0,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1, 1, 2, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C2, C..."
1,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 2, 0, 0, ...","[C2, C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C..."
2,"[2, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, ...","[C1, C1, C1, C2, C3, C3, C3, C3, C3, C3, C3, C..."
3,"[2, 3, 3, 3, 4, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","[C3, C3, C3, C3, C3, C1, C1, C1, C1, C1, C1, C..."
4,"[2, 0, 0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C4, C4, C4, C..."
...,...,...
39947,"[0, 1, 1, 2, 2, 1, 0, 0, 0, 1, 1, 2, 2, 1, 2, ...","[C1, C1, C1, C1, C1, C5, C6, C1, C1, C1, C1, C..."
39948,"[1, 2, 2, 1, 2, 3, 0, 0, 0, 0, 1, 1, 1, 2, 2, ...","[C1, C1, C1, C3, C3, C3, C1, C1, C1, C1, C1, C..."
39949,"[1, 1, 1, 2, 2, 2, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C..."
39950,"[2, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 1, 1, 1, 2, ...","[C1, C1, C1, C1, C1, C1, C1, C2, C4, C4, C3, C..."


In [164]:
log_label = []
for k in range(0,len(Thunderbird_log)-sequence_len,10):
    log_label.append(Thunderbird_log['Label'][k:k+sequence_len])
reshaped_label = np.array(log_label).astype('str')
result_label = []
for m in range(0,len(reshaped_label)):
    if '"VAPI' in reshaped_label[m]:
        result_label.append(1)
    else:
        result_label.append(0)
print(len(result_label))
print(result_label.count(1))
print(result_label.count(0))

# print(reshaped_label[0:10])
# print(Thunderbird_log[0:10])

39952
33885
6067


In [182]:
df_dict['Label']=result_label
df_dict

,EventSequence,Component,Label
0,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
1,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 2, 0, 0, ...","['C2', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
2,"[2, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, ...","['C1', 'C1', 'C1', 'C2', 'C3', 'C3', 'C3', 'C3...",1
3,"[2, 3, 3, 3, 4, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","['C3', 'C3', 'C3', 'C3', 'C3', 'C1', 'C1', 'C1...",1
4,"[2, 0, 0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
...,...,...,...
39947,"[0, 1, 1, 2, 2, 1, 0, 0, 0, 1, 1, 2, 2, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C5', 'C6', 'C1...",1
39948,"[1, 2, 2, 1, 2, 3, 0, 0, 0, 0, 1, 1, 1, 2, 2, ...","['C1', 'C1', 'C1', 'C3', 'C3', 'C3', 'C1', 'C1...",1
39949,"[1, 1, 1, 2, 2, 2, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
39950,"[2, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 1, 1, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C2...",1


In [183]:
df_dict.to_csv("Thunderbird_C_seq",index=False)

In [184]:
df_dict = pd.read_csv("Thunderbird_C_seq",memory_map=True)
df_dict

,EventSequence,Component,Label
0,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 2, 0, 0, 1, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
1,"[0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2, 2, 2, 0, 0, ...","['C2', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
2,"[2, 2, 2, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, ...","['C1', 'C1', 'C1', 'C2', 'C3', 'C3', 'C3', 'C3...",1
3,"[2, 3, 3, 3, 4, 0, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","['C3', 'C3', 'C3', 'C3', 'C3', 'C1', 'C1', 'C1...",1
4,"[2, 0, 0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
...,...,...,...
39947,"[0, 1, 1, 2, 2, 1, 0, 0, 0, 1, 1, 2, 2, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C5', 'C6', 'C1...",1
39948,"[1, 2, 2, 1, 2, 3, 0, 0, 0, 0, 1, 1, 1, 2, 2, ...","['C1', 'C1', 'C1', 'C3', 'C3', 'C3', 'C1', 'C1...",1
39949,"[1, 1, 1, 2, 2, 2, 0, 1, 1, 2, 2, 0, 0, 1, 1, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1...",1
39950,"[2, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 1, 1, 1, 2, ...","['C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C2...",1


In [191]:
df_dict["EventSequence"] = df_dict["EventSequence"].map(lambda x: eval(x))

In [193]:
df_dict["Component"] = df_dict["Component"].map(lambda x: eval(x))

In [194]:
for index,row in  df_dict.iterrows():
    for x in range(len(row["EventSequence"])):
        row["EventSequence"][x] = row["EventSequence"][x]+1
    row["EventSequence"].append(0)

In [195]:
for index,row in  df_dict.iterrows():
    row["Component"].append("C0")

In [196]:
df_dict

,EventSequence,Component,Label
0,"[1, 1, 2, 2, 3, 3, 1, 1, 2, 3, 1, 1, 2, 2, 3, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C2, C...",1
1,"[1, 1, 2, 2, 3, 3, 1, 1, 2, 2, 3, 3, 3, 1, 1, ...","[C2, C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C...",1
2,"[3, 3, 3, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, ...","[C1, C1, C1, C2, C3, C3, C3, C3, C3, C3, C3, C...",1
3,"[3, 4, 4, 4, 5, 1, 1, 2, 2, 3, 3, 1, 1, 2, 2, ...","[C3, C3, C3, C3, C3, C1, C1, C1, C1, C1, C1, C...",1
4,"[3, 1, 1, 2, 2, 2, 3, 3, 1, 1, 1, 1, 1, 1, 1, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C4, C4, C4, C...",1
...,...,...,...
39947,"[1, 2, 2, 3, 3, 2, 1, 1, 1, 2, 2, 3, 3, 2, 3, ...","[C1, C1, C1, C1, C1, C5, C6, C1, C1, C1, C1, C...",1
39948,"[2, 3, 3, 2, 3, 4, 1, 1, 1, 1, 2, 2, 2, 3, 3, ...","[C1, C1, C1, C3, C3, C3, C1, C1, C1, C1, C1, C...",1
39949,"[2, 2, 2, 3, 3, 3, 1, 2, 2, 3, 3, 1, 1, 2, 2, ...","[C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C1, C...",1
39950,"[3, 1, 1, 2, 2, 3, 3, 1, 1, 1, 1, 2, 2, 2, 3, ...","[C1, C1, C1, C1, C1, C1, C1, C2, C4, C4, C3, C...",1


In [197]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_index,test_index in split.split(df_dict,df_dict["Label"]):
    strat_train_set=df_dict.loc[train_index]
    strat_test_set=df_dict.loc[test_index]
strat_test_set["Label"].value_counts() / len(strat_test_set)# 计算分层抽样后的各个数据占的比例

1    0.848204
0    0.151796
Name: Label, dtype: float64

In [198]:
strat_train_set = df_dict[:33559]
strat_test_set = df_dict[33559:]

In [199]:
strat_test_set.reset_index(drop=True,inplace=True)

In [202]:
from dgl.data import DGLDataset
from dgl.data.utils import save_graphs
class TestDataset(DGLDataset):
    def __init__(self):
        super(TestDataset,self).__init__(name='TestDataset')
    def save(self):
        # save graphs and labels
        graph_path = 'Thunderbird_Test_seq.bin'
        save_graphs(graph_path, self.graphs, {'labels': self.label})
    def process(self):
        node_max = 0
        self.graphs=[]
        data_dict = strat_test_set
        self.label = torch.tensor(data_dict["Label"].values)
        for q in range(data_dict.shape[0]):
            Event = np.array(data_dict['EventSequence'][q])
            Componen = np.array(data_dict['Component'][q])
            graph_1 = {}
            CC = np.unique(Componen)
            for i in CC:
                s = [data_dict['EventSequence'][q][data_dict['Component'][q].index(i)]]
                t = []
                for j in range(data_dict['Component'][q].index(i)+1,len(data_dict['Component'][q])):
                    if data_dict['Component'][q][j] == i:
                        t.append(data_dict['EventSequence'][q][j])
                        s.append(data_dict['EventSequence'][q][j])
                s.pop()
                T = torch.IntTensor(t)
                S = torch.IntTensor(s)
                graph_1[(i, 'con', i)] = (S, T)
            graph_2 = {}
            for i in range(len(data_dict['EventSequence'][q])-1):
                if data_dict['Component'][q][i] != data_dict['Component'][q][i+1]:
                    l = [[data_dict['EventSequence'][q][i]],[data_dict['EventSequence'][q][i+1]]]
                    if (data_dict['Component'][q][i], 'dif', data_dict['Component'][q][i+1]) in graph_2:
                        graph_2[(data_dict['Component'][q][i], 'dif', data_dict['Component'][q][i+1])][0].append(data_dict['EventSequence'][q][i])
                        graph_2[(data_dict['Component'][q][i], 'dif', data_dict['Component'][q][i+1])][1].append(data_dict['EventSequence'][q][i+1])
                    else:
                        graph_2[(data_dict['Component'][q][i], 'dif', data_dict['Component'][q][i+1])] = l
            for key in graph_2:
                S = torch.IntTensor(graph_2[key][0])
                T = torch.IntTensor(graph_2[key][1])
                graph_2[key] = (S, T)
            graph = {}
            graph.update(graph_1) 
            graph.update(graph_2)
            G = dgl.heterograph(graph)
            sum =0
            for x in G.ntypes:
                sum =sum+G.number_of_nodes(x)
            if sum>node_max:
                node_max = sum
            self.graphs.append(G)
        print(node_max)
    def __getitem__(self, idx):
        return self.graphs[idx], self.label[idx]
    def __len__(self):
        return len(self.graphs)

In [203]:
Testdataset = TestDataset()

243


In [204]:
from dgl.data import DGLDataset
from dgl.data.utils import save_graphs
class TrainDataset(DGLDataset):
    def __init__(self):
        super(TrainDataset,self).__init__(name='TrainDataset')
    def save(self):
        # save graphs and labels
        graph_path = 'Thunderbird_Train_seq.bin'
        save_graphs(graph_path, self.graphs, {'labels': self.label})
    def process(self):
        self.graphs=[]
        node_max = 0
        data_df = strat_train_set
        self.label = torch.tensor(data_df["Label"].values)
        for q in range(data_df.shape[0]):
            Event = np.array(data_df['EventSequence'][q])
            Componen = np.array(data_df['Component'][q])
            graph_1 = {}
            CC = np.unique(Componen)
            for i in CC:
                s = [data_df['EventSequence'][q][data_df['Component'][q].index(i)]]
                t = []
                for j in range(data_df['Component'][q].index(i)+1,len(data_df['Component'][q])):
                    if data_df['Component'][q][j] == i:
                        t.append(data_df['EventSequence'][q][j])
                        s.append(data_df['EventSequence'][q][j])
                s.pop()
                T = torch.IntTensor(t)
                S = torch.IntTensor(s)
                graph_1[(i, 'con', i)] = (S, T)
            graph_2 = {}
            for i in range(len(data_df['EventSequence'][q])-1):
                if data_df['Component'][q][i] != data_df['Component'][q][i+1]:
                    l = [[data_df['EventSequence'][q][i]],[data_df['EventSequence'][q][i+1]]]
                    if (data_df['Component'][q][i], 'dif', data_df['Component'][q][i+1]) in graph_2:
                        graph_2[(data_df['Component'][q][i], 'dif', data_df['Component'][q][i+1])][0].append(data_df['EventSequence'][q][i])
                        graph_2[(data_df['Component'][q][i], 'dif', data_df['Component'][q][i+1])][1].append(data_df['EventSequence'][q][i+1])
                    else:
                        graph_2[(data_df['Component'][q][i], 'dif', data_df['Component'][q][i+1])] = l
            for key in graph_2:
                S = torch.IntTensor(graph_2[key][0])
                T = torch.IntTensor(graph_2[key][1])
                graph_2[key] = (S, T)
            graph = {}
            graph.update(graph_1) 
            graph.update(graph_2)
            G = dgl.heterograph(graph)
            sum =0
            for x in G.ntypes:
                sum =sum+G.number_of_nodes(x)
            if sum>node_max:
                node_max = sum
            self.graphs.append(G)
        print(node_max)
    def __getitem__(self, idx):
        return self.graphs[idx], self.label[idx]
    def __len__(self):
        return len(self.graphs)

In [205]:
Traindataset = TrainDataset()

247
